# 🧱 Part 19: CoT + Thinking Models: How To Make A Model “Think Before It Speaks”

> **Background**: A standard LLM often “answers immediately”. Ask it `357 x 289` and it may just guess a number based on pattern matching. Thinking models (o1, DeepSeek-R1, etc.) instead generate an internal scratchpad and then produce a final answer.
>
> **Goal for this part**: Understand the core idea behind Chain-of-Thought (CoT), why Self-Consistency works, what `<think>...</think>` really is (special tokens + training), and how modern thinking models are trained (cold-start SFT -> RL -> rejection sampling -> broader RL).

One-line core idea:

**Standard model = answer directly; thinking model = draft reasoning first, then answer.**

Run the cells in order. The code is deliberately small and explicit.


In [1]:
import numpy as np

np.random.seed(42)


### 1. Problem: why are LLMs so bad at math?

```text
User: 357 x 289 = ?

A standard LLM's internal flow:
  input tokens -> embeddings -> transformer x N -> output "103173"

It did not actually "compute".
It guessed based on patterns in training data.
If 357 x 289 never appeared in training, it will often be wrong.
```

**Root cause**: a Transformer has a fixed amount of compute per forward pass. Whether the question is `1+1` or calculus, the compute budget is the same. It has no built-in notion of multi-step reasoning.

Humans do multi-step arithmetic:

1. 357 x 200 = 71400
2. 357 x 80  = 28560
3. 357 x 9   = 3213
4. 71400 + 28560 + 3213 = 103173

LLMs need the same “step-by-step” structure. That is what CoT provides.


### 2. CoT (Chain-of-Thought): make the model write the reasoning steps

The idea is simple: **demonstrate the format “reasoning first, answer last” in the prompt**.

```text
No CoT prompt:
  Q: 357 x 289 = ?
  A: 103173  <- model guesses directly

CoT few-shot prompt:
  Q: 123 x 45 = ?
  A: 123x40=4920, 123x5=615, 4920+615=5535. Answer: 5535.

  Q: 357 x 289 = ?
  A:  <- the model imitates the format: steps first, then answer
```

Why does it help?

When the model generates intermediate steps, those intermediate results become **additional context** for later tokens. Attention can read the intermediate numbers and continue the reasoning.

In essence: **CoT turns a “one-pass” problem into a “many-token relay” problem.**


In [2]:
# A concrete demo: why CoT helps
print("=== Why CoT Helps ===")
print()

target = 357 * 289

# Without CoT: one-shot guessing
print("Without CoT:")
print("  The model has to output 357 x 289 in one step.")
print(f"  Correct answer: {target:,}")
np.random.seed(42)
guesses = np.random.randint(80000, 120000, size=5)
closest = guesses[np.argmin(np.abs(guesses - target))]
print(f"  A typical guess: {closest:,} (off by {abs(closest - target):,})")
print("  -> One-shot is hard.")
print()

# With CoT: decompose into steps
print("With CoT (step-by-step):")
result = 0
steps = []
for multiplier, label in [(200, "200"), (80, "80"), (9, "9")]:
    partial = 357 * multiplier
    result += partial
    steps.append(partial)
    print(f"  357 x {label} = {partial:,}")

total = sum(steps)
print(f"  {' + '.join(f'{s:,}' for s in steps)} = {total:,}")
print()
print(f"CoT result: {total:,} == correct answer {target:,}")
print()
print("Key idea: each intermediate result becomes context for later tokens.")
print("This trades generation time (more tokens) for effective compute depth.")


=== Why CoT Helps ===

Without CoT:
  The model has to output 357 x 289 in one step.
  Correct answer: 103,173
  A typical guess: 95,795 (off by 7,378)
  -> One-shot is hard.

With CoT (step-by-step):
  357 x 200 = 71,400
  357 x 80 = 28,560
  357 x 9 = 3,213
  71,400 + 28,560 + 3,213 = 103,173

CoT result: 103,173 == correct answer 103,173

Key idea: each intermediate result becomes context for later tokens.
This trades generation time (more tokens) for effective compute depth.


### 2.5 Self-Consistency: sample multiple reasoning paths and vote

CoT has a weakness: **a single reasoning chain can contain a mistake**.
Ask the same math problem 5 times (temperature > 0) and you may get multiple different chains and answers.

Self-Consistency:

```text
same question
  -> sample N different CoT reasoning chains
  -> vote on the final answer
  -> the most frequent answer wins
```

Why does it work?

Intuition: ask 5 classmates.

- Each may make a mistake at a different step.
- Different people tend to make different errors.
- The correct answer is unique, so it tends to appear most often.

A quick probability intuition: if one chain is correct with probability p, the probability that a majority of N chains are correct increases with N.

Self-Consistency requires **no retraining** and **no architecture change**. It is purely an inference-time trick: sample more and aggregate.


In [3]:
# ============================================================
# Self-Consistency demo: sample multiple reasoning chains and vote
# ============================================================
import random
random.seed(42)

print("=" * 70)
print("Problem: Ming has 15 apples, gives Hong 3, buys 8 more,")
print("         eats 2, then gives half of what remains to Gang.")
print("         How many apples does Ming have now?")
print("=" * 70)

reasoning_chains = [
    {
        "id": 1,
        "reasoning": [
            "Step 1: start with 15",
            "Step 2: give 3 -> 15 - 3 = 12",
            "Step 3: buy 8 -> 12 + 8 = 20",
            "Step 4: eat 2 -> 20 - 2 = 18",
            "Step 5: give half -> 18 / 2 = 9",
        ],
        "answer": 9,
        "correct": True,
    },
    {
        "id": 2,
        "reasoning": [
            "Step 1: start with 15",
            "Step 2: give 3 -> 12",
            "Step 3: buy 8 -> 20",
            "Step 4: eat 2 -> 18",
            "Step 5: give half -> 9",
        ],
        "answer": 9,
        "correct": True,
    },
    {
        "id": 3,
        "reasoning": [
            "Step 1: start with 15",
            "Step 2: give 3 -> 12",
            "Step 3: buy 8 -> 20",
            "Step 4: eat 2 -> 18",
            "Step 5: half of 18 is 9",
        ],
        "answer": 9,
        "correct": True,
    },
    {
        "id": 4,
        "reasoning": [
            "Step 1: start with 15",
            "Step 2: give 3 -> 12",
            "Step 3: buy 8 -> 20",
            "Step 4: eat 2 -> 18",
            "Step 5: (forgot to divide by 2) -> 18",
        ],
        "answer": 18,
        "correct": False,
    },
    {
        "id": 5,
        "reasoning": [
            "Step 1: 15 - 3 + 8 = 20",
            "Step 2: 20 - 2 = 18",
            "Step 3: give half -> 9 remain",
        ],
        "answer": 9,
        "correct": True,
    },
]

for chain in reasoning_chains:
    print()
    print(f"--- Chain #{chain['id']} ---")
    for step in chain['reasoning']:
        print(f"  {step}")
    status = "OK" if chain['correct'] else "WRONG"
    print(f"  Final answer: {chain['answer']}  [{status}]")

print()
print("=" * 70)
print("Voting")
print("=" * 70)

from collections import Counter
answers = [c['answer'] for c in reasoning_chains]
vote_counts = Counter(answers)

for ans, count in vote_counts.most_common():
    mark = "OK" if ans == 9 else "WRONG"
    bar = "█" * count
    print(f"  answer {ans}: {count} votes {bar}  [{mark}]")

winner = vote_counts.most_common(1)[0][0]
winner_is_correct = (winner == 9)
print()
print(f"Winner (majority vote): {winner}")
print(f"Correct? {'YES' if winner_is_correct else 'NO'}")

print()
print("=" * 70)
print("Single sample vs Self-Consistency")
print("=" * 70)
print("  Single sample (random pick): accuracy = 4/5 = 80%")
print("  Self-Consistency (vote over 5): accuracy = 100% (in this toy example)")
print()
print("Insight:")
print("  Chain #4 made a mistake (forgot to divide by 2).")
print("  The other chains agree on 9, so voting removes the outlier.")


Problem: Ming has 15 apples, gives Hong 3, buys 8 more,
         eats 2, then gives half of what remains to Gang.
         How many apples does Ming have now?

--- Chain #1 ---
  Step 1: start with 15
  Step 2: give 3 -> 15 - 3 = 12
  Step 3: buy 8 -> 12 + 8 = 20
  Step 4: eat 2 -> 20 - 2 = 18
  Step 5: give half -> 18 / 2 = 9
  Final answer: 9  [OK]

--- Chain #2 ---
  Step 1: start with 15
  Step 2: give 3 -> 12
  Step 3: buy 8 -> 20
  Step 4: eat 2 -> 18
  Step 5: give half -> 9
  Final answer: 9  [OK]

--- Chain #3 ---
  Step 1: start with 15
  Step 2: give 3 -> 12
  Step 3: buy 8 -> 20
  Step 4: eat 2 -> 18
  Step 5: half of 18 is 9
  Final answer: 9  [OK]

--- Chain #4 ---
  Step 1: start with 15
  Step 2: give 3 -> 12
  Step 3: buy 8 -> 20
  Step 4: eat 2 -> 18
  Step 5: (forgot to divide by 2) -> 18
  Final answer: 18  [WRONG]

--- Chain #5 ---
  Step 1: 15 - 3 + 8 = 20
  Step 2: 20 - 2 = 18
  Step 3: give half -> 9 remain
  Final answer: 9  [OK]

Voting
  answer 9: 4 votes ███

#### 2.5.1 Self-Consistency vs thinking models

Self-Consistency and thinking models (o1, R1) are different:

| | Self-Consistency | Thinking model |
|------|---------|-------|
| **How** | sample multiple chains + vote | learns internal self-checking during training |
| **Cost** | N-times inference cost | one inference call, but internal thinking can be long |
| **Training required** | no | yes (usually RL) |
| **Key capability** | diversity sampling | self-verification, correction, backtracking |

Conceptually:

- Self-Consistency is **external** error correction: you try multiple chains until one is right.
- Thinking models are **internal** error correction: the model checks itself inside one chain.

You can think of R1-style self-verification as a built-in, automatic form of Self-Consistency.


### 3. From CoT to thinking models: hide the scratchpad

With plain CoT, the reasoning trace is visible to the user. Many users do not want to see the scratch work.

Thinking models separate “scratchpad” and “final answer”:

```text
User-visible:
  Q: 357 x 289 = ?
  A: 103173

Model internally generates:
  <think>
  357x200=71400
  357x80=28560
  357x9=3213
  71400+28560+3213=103173
  </think>
  103173
```

`<think>` and `</think>` act like parentheses: the region inside is the draft, and the region after is the final answer.

Frontends can choose to hide or collapse the `<think>...</think>` region.

**Essence**: a thinking model is an engineered packaging of CoT. The reasoning is still there; it is just separated and managed by special tokens.


### 3.5 What is the `<think>` marker really doing?

#### 3.5.1 Why `<think>` should not be treated as normal text

If `<think>` is just ordinary characters, several problems appear:

1. Tokenization can split it: `<think>` may become `<`, `think`, `>` as separate tokens.
2. The closing boundary is unstable: the model might accidentally emit `</think>` inside the scratchpad and break the UI.
3. Training control is hard: you cannot reliably locate “thinking region vs answer region”.

So the robust approach is to add `<think>` and `</think>` as **special tokens**.

That means they get dedicated IDs, like `<BOS>` / `<EOS>`:

```text
<think>  -> 100
</think> -> 101
```

Now generating ID 100 means “enter thinking region”; generating ID 101 means “exit thinking region”.


#### 3.5.2 How do you add new thinking tokens in practice?

There are two typical cases.

**Case A: training a tokenizer + model from scratch**

Include the tokens in the tokenizer's special token list from the start:

```text
<BOS>, <EOS>, <PAD>, <think>, </think>
```

They will not be split by the tokenizer, and the model learns embeddings for them during training.

**Case B: continuing training from an existing base model (most common)**

Typical steps:

1. Add tokens to the tokenizer as special tokens (`<think>`, `</think>`).
2. Resize model embeddings and output layer to match the larger vocab.
3. Prepare formatted data that actually contains `<think>...</think>`.
4. Continue SFT and/or RL so the model learns when to open/close thinking.

Common pseudocode:

```python
new_tokens = {"additional_special_tokens": ["<think>", "</think>"]}
num_added = tokenizer.add_special_tokens(new_tokens)
model.resize_token_embeddings(len(tokenizer))

# then continue training on data that contains <think>...</think>
```

Important: **adding tokens is only giving the model a new pen**. It does not automatically make it reason. The reasoning behavior comes from data, masking choices, and reward design.


In [4]:
# A minimal example: adding <think> tokens creates a trainable boundary
vocab = {
    "<BOS>": 0,
    "<EOS>": 1,
    "<PAD>": 2,
    "user": 3,
    "assistant": 4,
    "answer": 5,
    "357": 6,
    "289": 7,
    "103173": 8,
}

new_symbols = ["<think>", "</think>"]
for symbol in new_symbols:
    if symbol not in vocab:
        vocab[symbol] = len(vocab)

train_tokens = [
    "<BOS>",
    "user", "357", "289",
    "assistant", "<think>", "357", "289", "103173", "</think>",
    "answer", "103173",
    "<EOS>",
]
train_ids = [vocab[token] for token in train_tokens]

print("New symbols:")
for symbol in new_symbols:
    print(f"  {symbol} -> ID {vocab[symbol]}")

print()
print("Training sample (tokens):")
print(train_tokens)
print()
print("Training sample (IDs):")
print(train_ids)
print()
print("Key observation: the model does not understand the English word 'think'.")
print("It learns that the region between IDs for <think> and </think> should contain a scratchpad.")


New symbols:
  <think> -> ID 9
  </think> -> ID 10

Training sample (tokens):
['<BOS>', 'user', '357', '289', 'assistant', '<think>', '357', '289', '103173', '</think>', 'answer', '103173', '<EOS>']

Training sample (IDs):
[0, 3, 6, 7, 4, 9, 6, 7, 8, 10, 5, 8, 1]

Key observation: the model does not understand the English word 'think'.
It learns that the region between IDs for <think> and </think> should contain a scratchpad.


In [5]:
# ============================================================
# Demo: why <think> should be a special token (tokenization behavior)
# ============================================================
print("=== Special token vs plain text: tokenization comparison ===")
print()

vocab = {
    "I": 1, "like": 2, "eat": 3, "apple": 4, "orange": 5,
    "think": 7, "<": 9, ">": 10, "/": 11,
    "reasoning": 13, "process": 14, "answer": 15, "is": 16,
}

print("Vocabulary (no special tokens):")
for k, v in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"  '{k}' -> {v}")
print()

print("=" * 55)
print("Scenario 1: <think> is plain text (not in vocab as a special token)")
print("=" * 55)

def tokenize_plain(text, vocab):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=len, reverse=True):
            if text[i:].startswith(word):
                matched = (word, vocab[word])
                break
        if matched:
            tokens.append(matched)
            i += len(matched[0])
        else:
            ch = text[i]
            tid = ord(ch) % 50 + 20
            tokens.append((ch if ch != ' ' else '⎵', tid))
            i += 1
    return tokens

text = "I like <think> reasoning process </think> answer is apple"
tokens_plain = tokenize_plain(text, vocab)

print(f"Input: {text}")
print(f"Tokenization ({len(tokens_plain)} tokens):")
ids = []
for word, tid in tokens_plain:
    ids.append(tid)
    flag = "(split tag)" if word in ("<", ">", "/") else ""
    print(f"  [{word:12s}] -> ID {tid:3d} {flag}")

print()
print(f"Token IDs: {ids}")
print()
print("Problems:")
print("  - '<think>' can become 3 tokens: '<' 'think' '>'")
print("  - '</think>' can be split too")
print("  - It's hard to reliably locate the scratchpad boundary by IDs")
print()

print("=" * 55)
print("Scenario 2: <think> is added as a special token (recommended)")
print("=" * 55)

vocab_with_special = vocab.copy()
vocab_with_special["<think>"] = 100
vocab_with_special["</think>"] = 101

print("Added tokens:")
print("  '<think>'  -> 100  (special)")
print("  '</think>' -> 101  (special)")
print()

def tokenize_with_special(text, vocab):
    tokens = []
    i = 0
    while i < len(text):
        matched = None
        for word in sorted(vocab.keys(), key=lambda x: (-len(x), x)):
            if text[i:].startswith(word):
                matched = (word, vocab[word])
                break
        if matched:
            tokens.append(matched)
            i += len(matched[0])
        else:
            ch = text[i]
            tokens.append((ch, ord(ch) % 50 + 20))
            i += 1
    return tokens

tokens_special = tokenize_with_special(text, vocab_with_special)

print(f"Input: {text}")
print(f"Tokenization ({len(tokens_special)} tokens):")
ids_special = []
for word, tid in tokens_special:
    ids_special.append(tid)
    flag = "(special)" if tid >= 100 else ""
    print(f"  [{word:12s}] -> ID {tid:3d} {flag}")

print()
print(f"Token IDs: {ids_special}")
print(f"Token count: {len(tokens_plain)} -> {len(tokens_special)}")
print()
print("Advantages:")
print("  - '<think>' is one token (ID=100)")
print("  - scratchpad boundaries are easy to locate (ID 100/101)")
print("  - the model only needs to learn when to emit token 100")

print()
print("Real tokenizer note:")
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    plain_result = enc.encode("<think>")
    print("  GPT-2 tokenizes '<think>' into multiple pieces:")
    print(f"    IDs: {plain_result}")
    print(f"    pieces: {[enc.decode([t]) for t in plain_result]}")
    print("  Thinking models add special tokens so '<think>' becomes a single ID.")
except ImportError:
    print("  (tiktoken not installed; skipping live demo)")
    print("  pip install tiktoken to run this section")

print()
print("Summary: add_special_tokens() is step 1 for training a thinking model.")


=== Special token vs plain text: tokenization comparison ===

Vocabulary (no special tokens):
  'I' -> 1
  'like' -> 2
  'eat' -> 3
  'apple' -> 4
  'orange' -> 5
  'think' -> 7
  '<' -> 9
  '>' -> 10
  '/' -> 11
  'reasoning' -> 13
  'process' -> 14
  'answer' -> 15
  'is' -> 16

Scenario 1: <think> is plain text (not in vocab as a special token)
Input: I like <think> reasoning process </think> answer is apple
Tokenization (22 tokens):
  [I           ] -> ID   1 
  [⎵           ] -> ID  52 
  [like        ] -> ID   2 
  [⎵           ] -> ID  52 
  [<           ] -> ID   9 (split tag)
  [think       ] -> ID   7 
  [>           ] -> ID  10 (split tag)
  [⎵           ] -> ID  52 
  [reasoning   ] -> ID  13 
  [⎵           ] -> ID  52 
  [process     ] -> ID  14 
  [⎵           ] -> ID  52 
  [<           ] -> ID   9 (split tag)
  [/           ] -> ID  11 (split tag)
  [think       ] -> ID   7 
  [>           ] -> ID  10 (split tag)
  [⎵           ] -> ID  52 
  [answer      ] -> ID  15 


### 3.6 How is loss computed when the model has a thinking region?

Once you introduce `<think>` tokens, a key question shows up:

**During training, do you compute loss on the thinking tokens?**

There are three common strategies. Consider one output token sequence:

```text
assistant tokens:
[<think>, 3, 1, 2, x, 2, 0, 0, =, 7, 1, 4, 0, 0, </think>, 1, 0, 3, 1, 7, 3]
|<------ thinking tokens ------->|<--- answer tokens --->|
```

Strategy comparison:

```text
+----------------------+-------------------+-------------------+----------------------+
|                      | A: Full loss      | B: Answer only    | C: Selective weight  |
+----------------------+-------------------+-------------------+----------------------+
| thinking tokens      | compute loss      | no loss           | loss but downweight  |
| answer tokens        | compute loss      | compute loss      | compute loss         |
| special tokens       | compute loss      | no loss           | no loss              |
+----------------------+-------------------+-------------------+----------------------+
| pros                 | simple; improves thinking quality | focuses on final answer | balanced             |
| cons                 | "performative" thinking risk      | thinking can degrade    | extra hyperparam     |
+----------------------+-------------------+-------------------+----------------------+
```

Why DeepSeek-R1 uses Full Loss (at the RL stage): if thinking tokens have no loss, the model is incentivized to write less thinking to get rewards faster, and thinking collapses.

Implementation-wise, you build a `loss_mask` (same shape as labels) with 0/1 weights and multiply per-token losses. The special tokens themselves typically do not contribute to loss.


In [6]:
# ============================================================
# Full implementation: build loss masks for three strategies
# ============================================================
import torch
import torch.nn.functional as F

print("=== Thinking Model Loss Mask: Reference Implementation ===\n")

THINK_START = 100
THINK_END = 101
IGNORE = -100

# Two toy label sequences in a batch
batch_labels = torch.tensor([
    [100,   3,   1,   2,  11,   2,   0,   0, 101,   1,   0,   3,   1,   7,   3,   0],
    [100,   5,  12,   6,  13,   3,   0, 101,   3,   0,   0,   0,   0,   0,   0,   0],
])  # shape: [2, 16]

VOCAB_SIZE = 200
logits = torch.randn(2, 16, VOCAB_SIZE)

print("Batch labels:")
print(batch_labels)
print()

print("=" * 60)
print("Strategy A: Full loss (everything except PAD)")
print("=" * 60)

def make_full_loss_mask(labels, pad_id=0):
    return (labels != pad_id).float()

mask_full = make_full_loss_mask(batch_labels)
print("loss_mask (0=ignore, 1=include):")
print(mask_full.int())
print()

loss_full = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    batch_labels.view(-1),
    ignore_index=0,
    reduction='none',
).view(2, 16)

total_full = loss_full.sum() / mask_full.sum()
print(f"Average loss (full): {total_full:.4f}")
print("-> thinking + answer tokens are both optimized")
print()

print("=" * 60)
print("Strategy B: Answer only (only tokens after </think>)")
print("=" * 60)

def make_answer_only_mask(labels, think_end=101, pad_id=0):
    mask = torch.zeros_like(labels, dtype=torch.float)
    for b in range(labels.shape[0]):
        end_positions = (labels[b] == think_end).nonzero(as_tuple=True)[0]
        if len(end_positions) > 0:
            end_pos = end_positions[0].item()
            for pos in range(end_pos + 1, labels.shape[1]):
                if labels[b, pos] == pad_id:
                    break
                mask[b, pos] = 1.0
    return mask

mask_answer_only = make_answer_only_mask(batch_labels)
print("loss_mask (0=ignore, 1=include):")
print(mask_answer_only.int())
print()

# Mask labels for CE
labels_masked_B = batch_labels.clone()
labels_masked_B[mask_answer_only == 0] = IGNORE

loss_answer_only = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    labels_masked_B.view(-1),
    ignore_index=IGNORE,
)
print(f"Average loss (answer only): {loss_answer_only:.4f}")
print("-> only answer tokens are optimized; thinking is unconstrained")
print()

print("=" * 60)
print("Strategy C: Selective weighting (thinking downweighted)")
print("=" * 60)

def make_selective_weight_mask(labels, think_start=100, think_end=101, thinking_weight=0.1, pad_id=0):
    weight_mask = torch.zeros_like(labels, dtype=torch.float)
    for b in range(labels.shape[0]):
        in_thinking = False
        for pos in range(labels.shape[1]):
            tid = labels[b, pos].item()
            if tid == think_start:
                in_thinking = True
                continue
            if tid == think_end:
                in_thinking = False
                continue
            if tid == pad_id:
                break
            weight_mask[b, pos] = thinking_weight if in_thinking else 1.0
    return weight_mask

weight_mask = make_selective_weight_mask(batch_labels)
print("Weight mask:")
for b in range(2):
    print(f"  sample {b}: {[f'{w:.1f}' for w in weight_mask[b].tolist()]}")
print()

loss_per_token = F.cross_entropy(
    logits.view(-1, VOCAB_SIZE),
    batch_labels.view(-1),
    ignore_index=0,
    reduction='none',
).view(2, 16)

weighted_loss = (loss_per_token * weight_mask).sum() / weight_mask.sum()
print(f"Average loss (selective weighting): {weighted_loss:.4f}")
print("-> thinking has 10% signal; answer has 100%")

print()
print("=" * 60)
print("Summary")
print("=" * 60)
print("- SFT stage often uses Answer Only (teach the format).")
print("- RL stage often uses Full Loss (DeepSeek-R1 style) to prevent thinking collapse.")
print("- Selective weighting can be a smooth transition.")
print("Common rule: special tokens (<think>, </think>) themselves usually do not receive loss.")


=== Thinking Model Loss Mask: Reference Implementation ===

Batch labels:
tensor([[100,   3,   1,   2,  11,   2,   0,   0, 101,   1,   0,   3,   1,   7,
           3,   0],
        [100,   5,  12,   6,  13,   3,   0, 101,   3,   0,   0,   0,   0,   0,
           0,   0]])

Strategy A: Full loss (everything except PAD)
loss_mask (0=ignore, 1=include):
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)

Average loss (full): 5.9277
-> thinking + answer tokens are both optimized

Strategy B: Answer only (only tokens after </think>)
loss_mask (0=ignore, 1=include):
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)

Average loss (answer only): 5.2270
-> only answer tokens are optimized; thinking is unconstrained

Strategy C: Selective weighting (thinking downweighted)
Weight mask:
  sample 0: ['0.0', '0.1', '0.1', '0.1', '0

### 4. How are thinking models trained?

A typical recipe has four steps:

```text
Step 1: Cold-start SFT
  Collect a few thousand high-quality examples with reasoning traces.
  Format: Q -> thinking -> A
  Supervised fine-tune so the model learns the "think then answer" format.

Step 2: RL for reasoning (the key step)
  Use reinforcement learning to improve reasoning ability.
  Reward signals often include:
    - correct answer: +1
    - wrong answer:  -1
    - correct formatting (<think> region): +0.1
    - language consistency: +0.1
  The model explores better reasoning strategies.

Step 3: Rejection sampling + SFT
  Use the RL-trained model to generate more Q->thinking->A data.
  Keep only samples with correct answers, then fine-tune again.

Step 4: Broad RL
  Run RL on broader tasks (helpfulness, safety, etc.) so the model is usable beyond math.
```

**Step 2 is the core**: RL lets the model discover reasoning strategies instead of memorizing human-written chains.


In [7]:
# Simulate how reward signals shape reasoning in RL
print("=== RL reasoning training (toy simulation) ===")
print()

print("Question: 15 + 27 = ?")
print()

attempts = [
    ("15+20=35, 35+7=42", 42, True),
    ("15+27=42", 42, True),
    ("15+30=45, 45-3=42", 42, True),
    ("Guess: 41", 41, False),
    ("10+20=30, 5+7=12, 30+12=42", 42, True),
]

for i, (reasoning, answer, correct) in enumerate(attempts):
    reward = 1 if correct else -1
    steps = len(reasoning.split(','))
    detail_bonus = min(steps * 0.05, 0.2)
    total_reward = reward + detail_bonus

    status = 'OK' if correct else 'WRONG'
    print(f"Attempt {i+1}: [{status}] {reasoning}")
    print(f"  answer={answer}, correct={correct}, base_reward={reward}, detail_bonus={detail_bonus:.2f}")
    print(f"  total_reward={total_reward:+.2f}")
    print()

print("RL reinforces high-reward reasoning patterns and suppresses low-reward ones.")


=== RL reasoning training (toy simulation) ===

Question: 15 + 27 = ?

Attempt 1: [OK] 15+20=35, 35+7=42
  answer=42, correct=True, base_reward=1, detail_bonus=0.10
  total_reward=+1.10

Attempt 2: [OK] 15+27=42
  answer=42, correct=True, base_reward=1, detail_bonus=0.05
  total_reward=+1.05

Attempt 3: [OK] 15+30=45, 45-3=42
  answer=42, correct=True, base_reward=1, detail_bonus=0.10
  total_reward=+1.10

Attempt 4: [WRONG] Guess: 41
  answer=41, correct=False, base_reward=-1, detail_bonus=0.05
  total_reward=-0.95

Attempt 5: [OK] 10+20=30, 5+7=12, 30+12=42
  answer=42, correct=True, base_reward=1, detail_bonus=0.15
  total_reward=+1.15

RL reinforces high-reward reasoning patterns and suppresses low-reward ones.


### 5. How to train your own thinking model

You do not need to train from scratch. You can fine-tune a strong open base model.

A practical workflow:

1. Choose a base model: Qwen2.5-7B / DeepSeek-V3 / Llama-3, etc. (the base must be decent at reasoning).
2. Prepare cold-start data (~3k-5k): generate with a strong model (R1 / GPT-4o) or extract from GSM8K/MATH.
3. Cold-start SFT (~1-2 hours on a single A100): teach the `<think>` + answer format.
4. RL training (~1-3 days): use frameworks like `verl` or OpenRLHF; reward correctness + format.
5. Rejection sampling + second SFT: generate more data, filter by correctness, fine-tune again.

Rough cost estimate for Qwen2.5-7B on 4x A100: a few days and a few hundred dollars.


In [8]:
# Example cold-start data format
print("=== Cold-start data format examples ===")
print()

training_examples = [
    {
        "question": "A rectangle is 12cm by 8cm. What is its area?",
        "thinking": "Area = length x width\\nArea = 12 x 8 = 96\\nSo the area is 96 square centimeters.",
        "answer": "96 square centimeters",
    },
    {
        "question": "Ming has 15 apples, eats 3, then buys 7 more. How many now?",
        "thinking": "Start: 15\\nEat 3: 15 - 3 = 12\\nBuy 7: 12 + 7 = 19\\nSo he has 19 apples.",
        "answer": "19",
    },
    {
        "question": "357 x 289 = ?",
        "thinking": "357 x 200 = 71400\\n357 x 80 = 28560\\n357 x 9 = 3213\\n71400 + 28560 + 3213 = 103173",
        "answer": "103173",
    },
]

for i, ex in enumerate(training_examples):
    print(f"--- sample {i+1} ---")
    print("<|user|>")
    print(ex['question'])
    print()
    print("<|assistant|><think>")
    print(ex['thinking'])
    print("</think>")
    print(ex['answer'])
    print()


=== Cold-start data format examples ===

--- sample 1 ---
<|user|>
A rectangle is 12cm by 8cm. What is its area?

<|assistant|><think>
Area = length x width\nArea = 12 x 8 = 96\nSo the area is 96 square centimeters.
</think>
96 square centimeters

--- sample 2 ---
<|user|>
Ming has 15 apples, eats 3, then buys 7 more. How many now?

<|assistant|><think>
Start: 15\nEat 3: 15 - 3 = 12\nBuy 7: 12 + 7 = 19\nSo he has 19 apples.
</think>
19

--- sample 3 ---
<|user|>
357 x 289 = ?

<|assistant|><think>
357 x 200 = 71400\n357 x 80 = 28560\n357 x 9 = 3213\n71400 + 28560 + 3213 = 103173
</think>
103173



### 6. The “aha moment” in RL: emergent self-reflection

DeepSeek-R1 reports an interesting phenomenon: during RL, the model learns to *reflect* on its own reasoning.

```text
thinking trace:
  Step 1: I think the answer is 42...
  Step 2: wait, let me check...
  Step 3: that's not right; I made a mistake...
  Step 4: recompute carefully -> 42. Confirmed.
```

**Nobody hard-coded that behavior.** RL only rewards correct outcomes. The model discovers that checking its own work increases rewards, so self-verification emerges.

This is the power of RL: you do not specify "how to think"; you only reward "thinking that ends up correct".


In [9]:
# Simulate emergent "self-reflection" behavior during RL
print("=== Emergent self-reflection in RL (toy) ===")
print()

def evaluate_reasoning(reasoning, correct_answer):
    # Return (parsed answer, is_correct, reward)
    import re

    numbers = re.findall(r'[\d.]+', reasoning)
    answer = float(numbers[-1]) if numbers else None
    correct = (answer == correct_answer)

    reward = 1.0 if correct else -1.0
    steps = reasoning.count('+') + reasoning.count('-') + reasoning.count('*') + reasoning.count('=')
    reward += min(steps * 0.05, 0.2)

    has_check = any(w in reasoning.lower() for w in [
        'verify', 'check', 'confirm', 'recheck', 'recompute', "that's wrong", 'wait',
    ])
    if has_check:
        reward += 0.1

    return answer, correct, reward

# Two candidate traces
traces = [
    "I think 15+27=41. Wait, let me check. 15+20=35, 35+7=42. Confirm: 42.",
    "15+27=41.",
]

for t in traces:
    ans, ok, r = evaluate_reasoning(t, correct_answer=42)
    print(f"Trace: {t}")
    print(f"  parsed_answer={ans}, correct={ok}, reward={r:+.2f}")
    print()

print("RL does not hard-code reflection, but it can reward it indirectly as a useful strategy.")


=== Emergent self-reflection in RL (toy) ===

Trace: I think 15+27=41. Wait, let me check. 15+20=35, 35+7=42. Confirm: 42.
  parsed_answer=42.0, correct=True, reward=+1.30

Trace: 15+27=41.
  parsed_answer=41.0, correct=False, reward=-0.90

RL does not hard-code reflection, but it can reward it indirectly as a useful strategy.


### 7. Limitations of thinking models

| Limitation | What it means |
|:---|:---|
| **Slow** | thinking traces can be hundreds or thousands of tokens |
| **Expensive** | thinking tokens also cost money in API billing |
| **Overthinking** | easy questions can trigger long reasoning |
| **Language mixing** | scratchpad may mix languages |
| **Hard to control** | RL-learned strategies are a black box |

Good for: math, programming, logic, scientific reasoning.

Not ideal for: casual chat, creative writing, emotional support.


### 8. A quick comparison of mainstream thinking models

| Model | Training recipe | Notes |
|:---|:---|:---|
| **OpenAI o1** | not fully public (likely RL + CoT) | strongest reasoning, closed source |
| **DeepSeek-R1** | cold-start SFT -> RL -> rejection sampling -> SFT -> broad RL | strongest open-source, detailed paper |
| **Qwen3** | similar pipeline | supports thinking / non-thinking dual modes |
| **Kimi K2** | not public | long context + reasoning |

Common pattern: RL is central, and models separate “thinking” vs “answer” phases.


### 9. Practical: how to enable / disable thinking across platforms

Different platforms expose thinking differently.

---

#### 9.1 OpenAI (o1, o3, o4-mini, GPT-5)

OpenAI thinking is controlled via an API parameter such as `reasoning_effort`:

```text
reasoning_effort = "low" | "medium" | "high"

low    -> less thinking, faster
medium -> balance
high   -> more thinking, best for hard reasoning
```

Key points:

- Only o-series models support this.
- The thinking content is hidden; you may see `reasoning_tokens` but not the raw trace.
- Billing counts reasoning tokens and output tokens (often priced differently).

---

#### 9.2 DeepSeek-R1

Open-source R1-style models typically use a chat template that separates `<think>...</think>` from the final answer. Many UIs collapse the `<think>` block.

DeepSeek-R1 usually cannot “turn off” thinking: it is a thinking-first model.

---

#### 9.3 Qwen3 (thinking / non-thinking dual mode)

Qwen3 is notable because it can switch thinking on/off at runtime, e.g. via `enable_thinking=True/False` in the chat template.

---

#### 9.4 Anthropic Claude (Extended Thinking)

Claude exposes a `thinking` configuration block with a token budget.

---

#### Summary table

| Platform / model | How to enable | Can disable? | Is thinking visible? |
|:---|:---|:---|:---|
| OpenAI o-series | `reasoning_effort` | adjustable | hidden |
| DeepSeek-R1 | always-on | no | returned separately by some APIs |
| Qwen3 | `enable_thinking=True/False` | yes | UI-controlled |
| Claude | `thinking.type="enabled"` | yes (omit) | separated |
| Local R1 distills | chat template + UI filtering | depends | UI-controlled |


In [10]:
# ============================================================
# Practical: API call examples for enabling thinking (runnable if keys are set)
# If keys are missing, this cell prints equivalent curl commands and skips.
# ============================================================

import os

# 1) OpenAI o-series (reasoning_effort)
openai_key = os.environ.get('OPENAI_API_KEY')
if openai_key:
    from openai import OpenAI
    client = OpenAI(api_key=openai_key)

    response = client.chat.completions.create(
        model='o1',
        messages=[{'role': 'user', 'content': '357 x 289 = ?'}],
        reasoning_effort='medium',
    )

    print("=== OpenAI result ===")
    print(response.choices[0].message.content)
else:
    print("=== OpenAI API (OPENAI_API_KEY not set; skipping) ===")
    print("Equivalent curl:")
    print('curl https://api.openai.com/v1/chat/completions \\\n  -H "Authorization: Bearer $OPENAI_API_KEY" \\\n  -H "Content-Type: application/json" \\\n  -d "{\\\"model\\\":\\\"o1\\\",\\\"reasoning_effort\\\":\\\"medium\\\",\\\"messages\\\":[{\\\"role\\\":\\\"user\\\",\\\"content\\\":\\\"357 x 289 = ?\\\"}]}"')

# 2) DeepSeek (reasoning_content)
deepseek_key = os.environ.get('DEEPSEEK_API_KEY')
if deepseek_key:
    from openai import OpenAI
    client = OpenAI(api_key=deepseek_key, base_url='https://api.deepseek.com')

    response = client.chat.completions.create(
        model='deepseek-reasoner',
        messages=[{'role': 'user', 'content': '357 x 289 = ?'}],
    )

    msg = response.choices[0].message
    reasoning = getattr(msg, 'reasoning_content', None)
    if reasoning:
        print()
        print("=== DeepSeek reasoning (first 200 chars) ===")
        print(reasoning[:200] + ('...' if len(reasoning) > 200 else ''))
    print("=== DeepSeek answer ===")
    print(msg.content)
else:
    print()
    print("=== DeepSeek API (DEEPSEEK_API_KEY not set; skipping) ===")

print()
print("=== Qwen3 note ===")
print("In Transformers, some Qwen chat templates support enable_thinking=True/False.")

print()
print("=== Claude note ===")
print("Claude exposes a thinking config with budget_tokens; see Anthropic docs.")


=== OpenAI API (OPENAI_API_KEY not set; skipping) ===
Equivalent curl:
curl https://api.openai.com/v1/chat/completions \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -H "Content-Type: application/json" \
  -d "{\"model\":\"o1\",\"reasoning_effort\":\"medium\",\"messages\":[{\"role\":\"user\",\"content\":\"357 x 289 = ?\"}]}"

=== DeepSeek API (DEEPSEEK_API_KEY not set; skipping) ===

=== Qwen3 note ===
In Transformers, some Qwen chat templates support enable_thinking=True/False.

=== Claude note ===
Claude exposes a thinking config with budget_tokens; see Anthropic docs.


### 10. Train your own thinking model (step-by-step practical recipe)

This is a concrete end-to-end recipe based on Qwen2.5-7B. The goal is to train a model that reliably “thinks before it answers”.

---

#### Step 1: environment setup

```bash
conda create -n thinking-train python=3.10 -y
conda activate thinking-train

pip install torch==2.4.0 transformers datasets accelerate
pip install vllm

# pick a training framework
pip install llama-factory
# or
pip install axolotl
```

---

#### Step 2: prepare cold-start data (~3000-5000 samples)

Cold-start data is a triplet: **question -> thinking trace -> answer**.

Data sources:

1. Generate with a strong model (DeepSeek-R1 / GPT-4o).
2. Extract from public datasets: GSM8K (grade-school math), MATH (contest math), APPS (programming).
3. Mix languages if you care about multilingual behavior.

Example format (Qwen / DeepSeek style):

```json
[
  {
    "messages": [
      {"role": "user", "content": "A tank fills in 3 hours and drains in 5 hours. If both pipes are open, how long to fill?"},
      {"role": "assistant", "content": "<think>
fill rate = 1/3 tank/hour
drain rate = 1/5 tank/hour
net rate = 1/3 - 1/5 = 2/15 tank/hour
time = 1 / (2/15) = 15/2 = 7.5 hours
</think>
7.5 hours"}
    ]
  }
]
```

Guideline: keep thinking detailed but not excessive (avoid teaching the model to overthink trivial questions).

---

#### Step 3: cold-start SFT (teach the format)

Use LLaMA-Factory or similar tools to train a LoRA on the cold-start dataset.

---

#### Step 4: RL reasoning training (the step that makes it smarter)

Use `verl` (DeepSeek-style) or OpenRLHF. Reward correctness strongly and formatting lightly.

---

#### Step 5: rejection sampling + second SFT

Generate more data, keep only correct samples, then fine-tune again.

---

#### Time / cost estimate

| Stage | Hardware | Time | Cost (cloud) |
|:---|:---|:---|:---|
| cold-start SFT | 1x A100 (80G) | 1-2 hours | ~$3 |
| RL reasoning | 4x A100 (80G) | 1-3 days | ~$150-$400 |
| rejection + SFT | 1x A100 (80G) | 2-4 hours | ~$6 |
| **Total** | - | **2-4 days** | **~$200-$500** |

---

#### Common pitfalls

1. Thinking traces too short -> model learns formatting but not reasoning.
2. Reward only correctness -> model may “cheat” by outputting answers without thinking.
3. Training set too easy -> RL converges but reasoning does not improve.
4. Train/test leakage -> memorization.
5. Wrong chat template -> thinking tokens render incorrectly.

---

#### Shortcut: use an R1 distill model

If you only want to use a thinking model, use a released R1 distill checkpoint (e.g. DeepSeek-R1-Distill-Qwen-7B). It already learned the `<think>` convention.


In [11]:
# ============================================================
# Practical: training-script templates (runnable skeletons)
# ============================================================

import json
import os


def generate_coldstart_data(questions, output_path='coldstart_data.json'):
    # Batch-generate Q -> thinking -> answer samples via DeepSeek-R1 API.
    deepseek_key = os.environ.get('DEEPSEEK_API_KEY')
    if not deepseek_key:
        print('SKIP: set DEEPSEEK_API_KEY to call the API')
        return []

    from openai import OpenAI
    client = OpenAI(api_key=deepseek_key, base_url='https://api.deepseek.com')

    system_prompt = """You are a math reasoning assistant. For each question:
1) write detailed step-by-step reasoning
2) explain each step
3) end with a short final answer

Format MUST be:
<think>
(reasoning)
</think>
(final answer)
"""

    dataset = []
    for i, question in enumerate(questions):
        print(f'[{i+1}/{len(questions)}] {question[:50]}...')
        try:
            response = client.chat.completions.create(
                model='deepseek-reasoner',
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': question},
                ],
            )
            msg = response.choices[0].message
            thinking = getattr(msg, 'reasoning_content', '') or ''
            answer = msg.content or ''

            content = f"<think>\n{thinking}\n</think>\n{answer}"
            dataset.append({
                'messages': [
                    {'role': 'user', 'content': question},
                    {'role': 'assistant', 'content': content},
                ]
            })
        except Exception as e:
            print(f'  failed: {e}')

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
    print(f'Saved {len(dataset)} samples to {output_path}')
    return dataset


SAMPLE_QUESTIONS = [
    'A rectangle is 12cm by 8cm. What is its area?',
    'Ming has 15 apples, eats 3, buys 7 more. How many now?',
    '357 x 289 = ?',
    'A tank fills in 3 hours and drains in 5 hours. If both pipes are open, how long to fill?',
    'In the arithmetic sequence 3, 7, 11, ..., what is the 20th term?',
]

if os.environ.get('DEEPSEEK_API_KEY'):
    generate_coldstart_data(SAMPLE_QUESTIONS[:2])
else:
    print('generate_coldstart_data() is defined.')
    print('Set DEEPSEEK_API_KEY and call generate_coldstart_data(questions) to run it.')
    print('\nExample data structure:')
    example = {
        'messages': [
            {'role': 'user', 'content': '357 x 289 = ?'},
            {'role': 'assistant', 'content': '<think>...reasoning...</think>\n103173'},
        ]
    }
    print(json.dumps(example, indent=2))


generate_coldstart_data() is defined.
Set DEEPSEEK_API_KEY and call generate_coldstart_data(questions) to run it.

Example data structure:
{
  "messages": [
    {
      "role": "user",
      "content": "357 x 289 = ?"
    },
    {
      "role": "assistant",
      "content": "<think>...reasoning...</think>\n103173"
    }
  ]
}


---

## CoT + Thinking Models: Summary

1. CoT = make the model write reasoning steps; intermediate results become context for later reasoning.
2. Thinking models = an engineered packaging of CoT: special tokens hide the scratchpad.
3. Common training recipe: cold-start SFT -> RL reasoning -> rejection sampling -> broader RL.
4. RL is the key: you do not specify how to think; you reward being correct.
5. Self-reflection can emerge during RL as a successful strategy.
6. Each platform exposes “thinking mode” differently (OpenAI: `reasoning_effort`; DeepSeek: always-on; Qwen3: `enable_thinking`; Claude: `thinking.type`).
7. Qwen3 is a mainstream model that can switch thinking on/off at runtime.
8. Training your own thinking model is feasible on a small cluster with a few days and a few hundred dollars.

One sentence: standard models answer immediately; thinking models draft first and answer second, and RL is what teaches that drafting skill.
